# 🚕💱 Member 2 — Extract: NYC Taxi Parquet + Exchange Rates API
**Business Goal:** Load real trip data and live currency rates for profitability analysis.

**Pipeline Role:** SOURCE 2 + SOURCE 3 → saves `taxi_raw.parquet` and `fx_rates_raw.parquet`

In [1]:
import pandas as pd
import requests
import os
import json
from datetime import datetime

print('✅ Libraries loaded')
print(f'📅 Run time: {datetime.now()}')

# ── CONFIG ────────────────────────────────────────────────────────
TAXI_PATH   = r'C:\Users\hp\Desktop\Pipeline Flow\data\raw\yellow_tripdata_2024-01.parquet'
FX_API_KEY  = '1c8e87a46f394ce88e8525a67af94523'
FX_URL      = f'https://openexchangerates.org/api/latest.json?app_id={FX_API_KEY}'

os.makedirs('data/raw', exist_ok=True)
print('📁 Folders ready')

✅ Libraries loaded
📅 Run time: 2026-05-12 12:16:41.817672
📁 Folders ready


## 🚕 PART A — NYC Taxi Parquet File

In [2]:
# ── EXTRACT: Load Parquet File ─────────────────────────────────────
print('📂 Loading NYC Taxi Parquet file...')
df_taxi_full = pd.read_parquet(TAXI_PATH)

print(f'✅ Loaded {len(df_taxi_full):,} rows')
print(f'📊 Columns: {list(df_taxi_full.columns)}')
print(f'\n🔍 Sample data:')
print(df_taxi_full.head(3).to_string())

📂 Loading NYC Taxi Parquet file...
✅ Loaded 2,964,624 rows
📊 Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']

🔍 Sample data:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  total_amount  congestion_surcharge  Airport_fee
0         2  2024-01-01 00:57:55   2024-01-01 01:17:43              1.0           1.72         1.0                  N           186            79             2         17.7    1.0      0.5        0.00           0.0                    1.0         22.70                   2.5          0.0
1         1  

In [3]:
# ── SELECT: Keep only relevant columns ────────────────────────────
KEEP_COLS = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'passenger_count',
    'trip_distance',
    'fare_amount',
    'tip_amount',
    'total_amount',
    'payment_type',
    'PULocationID',
    'DOLocationID'
]

# Only keep columns that exist in the file
available_cols = [c for c in KEEP_COLS if c in df_taxi_full.columns]
df_taxi = df_taxi_full[available_cols].copy()

print(f'✅ Selected {len(available_cols)} columns')
print(f'📊 Shape: {df_taxi.shape}')

✅ Selected 10 columns
📊 Shape: (2964624, 10)


In [4]:
# ── QUALITY CHECK: Understand raw data ────────────────────────────
print('🔍 RAW DATA QUALITY REPORT:')
print(f'  Total rows:           {len(df_taxi):,}')
print(f'  Null values:\n{df_taxi.isnull().sum()}')
print(f'\n  Fare amount stats:')
print(df_taxi['fare_amount'].describe())
print(f'\n  Trip distance stats:')
print(df_taxi['trip_distance'].describe())

# Flag obvious bad data
negative_fares = (df_taxi['fare_amount'] < 0).sum()
zero_distance  = (df_taxi['trip_distance'] == 0).sum()
print(f'\n  ⚠️  Negative fares:    {negative_fares:,}')
print(f'  ⚠️  Zero distance trips: {zero_distance:,}')
print('  → These will be cleaned in Member 3 (PySpark Transform)')

🔍 RAW DATA QUALITY REPORT:
  Total rows:           2,964,624
  Null values:
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          140162
trip_distance                 0
fare_amount                   0
tip_amount                    0
total_amount                  0
payment_type                  0
PULocationID                  0
DOLocationID                  0
dtype: int64

  Fare amount stats:
count    2.964624e+06
mean     1.817506e+01
std      1.894955e+01
min     -8.990000e+02
25%      8.600000e+00
50%      1.280000e+01
75%      2.050000e+01
max      5.000000e+03
Name: fare_amount, dtype: float64

  Trip distance stats:
count    2.964624e+06
mean     3.652169e+00
std      2.254626e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.680000e+00
75%      3.110000e+00
max      3.127223e+05
Name: trip_distance, dtype: float64

  ⚠️  Negative fares:    37,448
  ⚠️  Zero distance trips: 60,371
  → These will be cleaned in Member 3 (PySpark Transform

In [5]:
# ── SAVE: Taxi raw data ───────────────────────────────────────────
TAXI_OUT = 'data/raw/taxi_raw.parquet'
df_taxi.to_parquet(TAXI_OUT, index=False)

size_mb = os.path.getsize(TAXI_OUT) / (1024*1024)
print(f'✅ Saved taxi_raw.parquet')
print(f'   Rows: {len(df_taxi):,}')
print(f'   Size: {size_mb:.1f} MB')

✅ Saved taxi_raw.parquet
   Rows: 2,964,624
   Size: 54.9 MB


## 💱 PART B — Open Exchange Rates API

In [6]:
# ── EXTRACT: Call Exchange Rates API ──────────────────────────────
print('🌐 Calling Open Exchange Rates API...')
response = requests.get(FX_URL, timeout=30)

if response.status_code == 200:
    fx_data = response.json()
    print(f'✅ Success!')
    print(f'   Base currency:  {fx_data["base"]}')
    print(f'   Total rates:    {len(fx_data["rates"])}')
    print(f'   Timestamp:      {datetime.fromtimestamp(fx_data["timestamp"])}')
else:
    raise Exception(f'❌ API Error: {response.status_code} — {response.text}')

🌐 Calling Open Exchange Rates API...
✅ Success!
   Base currency:  USD
   Total rates:    172
   Timestamp:      2026-05-12 12:00:00


In [7]:
# ── PARSE: Flatten rates into DataFrame ───────────────────────────
rates = fx_data['rates']

df_fx = pd.DataFrame([
    {
        'currency_code': code,
        'rate_to_usd':   rate,
        'usd_to_local':  rate,
        'base_currency': fx_data['base'],
        'rate_date':     datetime.fromtimestamp(fx_data['timestamp']).date().isoformat(),
        'extracted_at':  datetime.now().isoformat()
    }
    for code, rate in rates.items()
])

print(f'✅ Parsed {len(df_fx)} currency rates')

# Preview key currencies
key_currencies = ['ETB', 'EUR', 'GBP', 'JPY', 'CNY', 'INR', 'BRL', 'NGN']
print('\n💰 Key Currency Rates (vs USD):')
print(df_fx[df_fx['currency_code'].isin(key_currencies)][['currency_code','rate_to_usd']].to_string(index=False))

✅ Parsed 172 currency rates

💰 Key Currency Rates (vs USD):
currency_code  rate_to_usd
          BRL     4.909600
          CNY     6.792100
          ETB   157.000000
          EUR     0.851752
          GBP     0.738993
          INR    95.670346
          JPY   157.593500
          NGN  1371.890000


In [8]:
# ── SAVE: FX rates raw data ───────────────────────────────────────
FX_OUT = 'data/raw/fx_rates_raw.parquet'
df_fx.to_parquet(FX_OUT, index=False)

print(f'✅ Saved fx_rates_raw.parquet')
print(f'   Rows: {len(df_fx)}')
print(f'   Size: {os.path.getsize(FX_OUT) / 1024:.1f} KB')

print('\n📦 MEMBER 2 SUMMARY:')
print(f'  ✅ taxi_raw.parquet     → {len(df_taxi):,} trip records')
print(f'  ✅ fx_rates_raw.parquet → {len(df_fx)} currency rates')
print('\n🏁 Member 2 COMPLETE — both files ready for Member 3 (PySpark Transform)')

✅ Saved fx_rates_raw.parquet
   Rows: 172
   Size: 8.2 KB

📦 MEMBER 2 SUMMARY:
  ✅ taxi_raw.parquet     → 2,964,624 trip records
  ✅ fx_rates_raw.parquet → 172 currency rates

🏁 Member 2 COMPLETE — both files ready for Member 3 (PySpark Transform)
